### Transform sprints Data

In [0]:
%run ../00-common/01-environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

#### 1. Read data, select columns, standardize and rename.

In [0]:
display(spark.read.table(bronze_table))

In [0]:
sprints_df = (
    spark.read.table(bronze_table)
    .drop("url")
    .withColumnsRenamed({
    "raceName": "race_name",
    "constructorId": "constructor_id",
    "driverId": "driver_id",
    "positionText": "finish_position_text",
    "date": "race_date",
    "grid": "grid_position",
    "laps" : "completed_laps",
    "number" : "car_number",
    "position" : "finish_position",})
    );
display(sprints_df)

#### 2. Bussiness Keys validation and removcing duplicate rows

In [0]:
from pyspark.sql import functions as F
sprints_valid_df = (
    sprints_df.filter(
    F.col("season").isNotNull() &
    F.col("round").isNotNull() &
    F.col("constructor_id").isNotNull() &
    F.col("driver_id").isNotNull())
    .dropDuplicates(['season', 'round', 'constructor_id', 'driver_id'])
    )

display(sprints_valid_df)

### 3. Transform Values of Columns race_name to Title Case

In [0]:
sprints_final_df = sprints_valid_df.withColumns({
    "race_name": F.initcap(F.col("race_name")),
})

display(sprints_final_df)

### 4. Writing data to bronze table

In [0]:
(
    sprints_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)
display(spark.table(silver_table))